In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import ParameterGrid


In [2]:
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

print(f"Train: {train_df.shape}")
print(f"Val:   {val_df.shape}")
print(f"Test:  {test_df.shape}")


Train: (129477, 46)
Val:   (43755, 46)
Test:  (12853, 46)


In [3]:
log_q1 = train_df["log_price"].quantile(0.25)
log_q3 = train_df["log_price"].quantile(0.75)
log_iqr = log_q3 - log_q1

PRICE_FLOOR = np.exp(log_q1 - 3 * log_iqr)
PRICE_CAP = np.exp(log_q3 + 4 * log_iqr)

ratio_lower = train_df["price_ratio"].quantile(0.005)
ratio_upper = train_df["price_ratio"].quantile(0.9995)

train_data_mask = (
    (train_df["ClosePrice"] >= PRICE_FLOOR)
    & (train_df["ClosePrice"] <= PRICE_CAP)
    & (train_df["price_ratio"] >= ratio_lower)
    & (train_df["price_ratio"] <= ratio_upper)
)
val_data_mask = (
    (val_df["ClosePrice"] >= PRICE_FLOOR)
    & (val_df["ClosePrice"] <= PRICE_CAP)
    & (val_df["price_ratio"] >= ratio_lower)
    & (val_df["price_ratio"] <= ratio_upper)
)
test_data_mask = (
    (test_df["ClosePrice"] >= PRICE_FLOOR)
    & (test_df["ClosePrice"] <= PRICE_CAP)
    & (test_df["price_ratio"] >= ratio_lower)
    & (test_df["price_ratio"] <= ratio_upper)
)

train_df = train_df[train_data_mask].reset_index(drop=True)
val_df = val_df[val_data_mask].reset_index(drop=True)
test_df = test_df[test_data_mask].reset_index(drop=True)


In [4]:
TARGET = "log_price"
DROP_COLS = ["ClosePrice", "CloseDate", "price_ratio", TARGET]
feature_cols = [c for c in train_df.columns if c not in DROP_COLS]

X_train, y_train = train_df[feature_cols], train_df[TARGET]
X_val, y_val = val_df[feature_cols], val_df[TARGET]
X_test, y_test = test_df[feature_cols], test_df[TARGET]

RANDOM_STATE = 20260805

In [5]:
def evaluate(y_true_log, y_pred_log, label=""):
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)

    rmse = root_mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(
        f"{label:>10} | RMSE: ${rmse:,.0f}  MAE: ${mae:,.0f}  R2: {r2:.4f}  MAPE: {mape:.2f}%"
    )
    return {"rmse": rmse, "mae": mae, "r2": r2, "mape": mape}


In [6]:
X_train_no_outliers = X_train
y_train_no_outliers = y_train

X_val_no_outliers = X_val
y_val_no_outliers = y_val

X_test_no_outliers = X_test
y_test_no_outliers = y_test

print(f"X_train_no_outliers shape: {X_train_no_outliers.shape}")
print(f"X_val_no_outliers shape:   {X_val_no_outliers.shape}")
print(f"X_test_no_outliers shape:  {X_test_no_outliers.shape}")


X_train_no_outliers shape: (128729, 42)
X_val_no_outliers shape:   (43495, 42)
X_test_no_outliers shape:  (12790, 42)


## XGBoost

## Baseline with and without outliers

In [7]:
xgb_base = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_base.fit(X_train, y_train)
evaluate(y_val, xgb_base.predict(X_val), "XGB base")


  XGB base | RMSE: $517,990  MAE: $178,128  R2: 0.8558  MAPE: 11.85%


{'rmse': 517989.53828189377,
 'mae': 178128.0061750776,
 'r2': 0.8557938126725733,
 'mape': np.float64(11.854930786877935)}

In [8]:
xgb_base_no_outliers = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_base_no_outliers.fit(X_train_no_outliers, y_train_no_outliers)
evaluate(
    y_val_no_outliers,
    xgb_base_no_outliers.predict(X_val_no_outliers),
    "XGB base without outliers",
)


XGB base without outliers | RMSE: $517,990  MAE: $178,128  R2: 0.8558  MAPE: 11.85%


{'rmse': 517989.53828189377,
 'mae': 178128.0061750776,
 'r2': 0.8557938126725733,
 'mape': np.float64(11.854930786877935)}

### hyperparameter tuning

In [9]:
param_grid = {
    "max_depth": [4, 5, 6, 7],
    "learning_rate": [0.05, 0.1, 0.15],
    "n_estimators": [300, 600],
}


def grid_search_xgb(param_grid, X_train, y_train, X_val, y_val):
    results = []
    for params in ParameterGrid(param_grid):
        model = xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
        model.fit(X_train, y_train)
        pred = model.predict(X_val)
        r2 = r2_score(np.exp(y_val), np.exp(pred))
        results.append({**params, "val_r2": r2})
    return (
        pd.DataFrame(results)
        .sort_values("val_r2", ascending=False)
        .reset_index(drop=True)
    )


xgb_results = grid_search_xgb(
    param_grid,
    X_train_no_outliers,
    y_train_no_outliers,
    X_val_no_outliers,
    y_val_no_outliers,
)
xgb_results


,learning_rate,max_depth,n_estimators,val_r2
0,0.10,6,600,0.862863
1,0.15,6,600,0.860826
2,0.15,6,300,0.859898
3,0.10,7,600,0.859675
4,0.15,7,300,0.858388
5,0.15,7,600,0.857746
6,0.05,7,600,0.857258
7,0.10,7,300,0.857023
8,0.10,5,600,0.856755
9,0.10,6,300,0.855794


In [10]:
comparison_results = {}

comparison_results["Baseline (with outliers)"] = evaluate(
    y_val, xgb_base.predict(X_val), "XGB base w/ outliers"
)

comparison_results["Baseline (no outliers)"] = evaluate(
    y_val_no_outliers, xgb_base_no_outliers.predict(X_val_no_outliers), "XGB base clean"
)

best_params = xgb_results.iloc[0][
    ["max_depth", "learning_rate", "n_estimators"]
].to_dict()
best_params["max_depth"] = int(best_params["max_depth"])
best_params["n_estimators"] = int(best_params["n_estimators"])

xgb_tuned = xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, **best_params)
xgb_tuned.fit(X_train_no_outliers, y_train_no_outliers)
comparison_results["Tuned (no outliers)"] = evaluate(
    y_val_no_outliers, xgb_tuned.predict(X_val_no_outliers), "XGB tuned"
)


XGB base w/ outliers | RMSE: $517,990  MAE: $178,128  R2: 0.8558  MAPE: 11.85%
XGB base clean | RMSE: $517,990  MAE: $178,128  R2: 0.8558  MAPE: 11.85%
 XGB tuned | RMSE: $505,134  MAE: $172,691  R2: 0.8629  MAPE: 11.49%


In [11]:
comparison_df = pd.DataFrame(comparison_results).T
comparison_df["rmse"] = comparison_df["rmse"].map(lambda x: f"${x:,.0f}")
comparison_df["mae"] = comparison_df["mae"].map(lambda x: f"${x:,.0f}")
comparison_df["r2"] = comparison_df["r2"].map(lambda x: f"{x:.4f}")
comparison_df["mape"] = comparison_df["mape"].map(lambda x: f"{x:.2f}%")
comparison_df


,rmse,mae,r2,mape
Baseline (with outliers),"$517,990","$178,128",0.8558,11.85%
Baseline (no outliers),"$517,990","$178,128",0.8558,11.85%
Tuned (no outliers),"$505,134","$172,691",0.8629,11.49%


## LightGBM

In [12]:
lgb_base = lgb.LGBMRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
lgb_base.fit(X_train, y_train)
evaluate(y_val, lgb_base.predict(X_val), "LGBM base")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004018 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3472
[LightGBM] [Info] Number of data points in the train set: 128729, number of used features: 35
[LightGBM] [Info] Start training from score 13.779528
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

{'rmse': 524029.2162072377,
 'mae': 182287.11024839166,
 'r2': 0.8524113638593709,
 'mape': np.float64(12.119368032330227)}

In [13]:
lgb_base_no_outliers = lgb.LGBMRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
lgb_base_no_outliers.fit(X_train_no_outliers, y_train_no_outliers)
evaluate(
    y_val_no_outliers,
    lgb_base_no_outliers.predict(X_val_no_outliers),
    "LGBM base without outliers",
)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003829 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3472
[LightGBM] [Info] Number of data points in the train set: 128729, number of used features: 35
[LightGBM] [Info] Start training from score 13.779528
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

{'rmse': 524029.2162072377,
 'mae': 182287.11024839166,
 'r2': 0.8524113638593709,
 'mape': np.float64(12.119368032330227)}

In [14]:
def grid_search_lgb(param_grid, X_train, y_train, X_val, y_val):
    results = []
    for params in ParameterGrid(param_grid):
        model = lgb.LGBMRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
        model.fit(X_train, y_train)
        pred = model.predict(X_val)
        r2 = r2_score(np.exp(y_val), np.exp(pred))
        results.append({**params, "val_r2": r2})
    return (
        pd.DataFrame(results)
        .sort_values("val_r2", ascending=False)
        .reset_index(drop=True)
    )


param_grid = {
    "max_depth": [4, 5, 6, 7],
    "learning_rate": [0.05, 0.1, 0.15],
    "n_estimators": [300, 600],
}

lgb_results = grid_search_lgb(
    param_grid,
    X_train_no_outliers,
    y_train_no_outliers,
    X_val_no_outliers,
    y_val_no_outliers,
)
lgb_results


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003974 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3472
[LightGBM] [Info] Number of data points in the train set: 128729, number of used features: 35
[LightGBM] [Info] Start training from score 13.779528
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

,learning_rate,max_depth,n_estimators,val_r2
0,0.15,7,600,0.860247
1,0.15,6,600,0.859676
2,0.10,7,600,0.859517
3,0.10,6,600,0.858067
4,0.15,5,600,0.857820
5,0.10,5,600,0.857614
6,0.15,7,300,0.856742
7,0.15,4,600,0.855778
8,0.15,6,300,0.854960
9,0.15,5,300,0.853647


In [15]:
comparison_results_lgb = {}

comparison_results_lgb["Baseline (with outliers)"] = evaluate(
    y_val, lgb_base.predict(X_val), "LGBM base w/ outliers"
)

comparison_results_lgb["Baseline (no outliers)"] = evaluate(
    y_val_no_outliers,
    lgb_base_no_outliers.predict(X_val_no_outliers),
    "LGBM base clean",
)

best_params_lgb = lgb_results.iloc[0][
    ["max_depth", "learning_rate", "n_estimators"]
].to_dict()
best_params_lgb["max_depth"] = int(best_params_lgb["max_depth"])
best_params_lgb["n_estimators"] = int(best_params_lgb["n_estimators"])

lgb_tuned = lgb.LGBMRegressor(random_state=RANDOM_STATE, n_jobs=-1, **best_params_lgb)
lgb_tuned.fit(X_train_no_outliers, y_train_no_outliers)
comparison_results_lgb["Tuned (no outliers)"] = evaluate(
    y_val_no_outliers, lgb_tuned.predict(X_val_no_outliers), "LGBM tuned"
)


LGBM base w/ outliers | RMSE: $524,029  MAE: $182,287  R2: 0.8524  MAPE: 12.12%
LGBM base clean | RMSE: $524,029  MAE: $182,287  R2: 0.8524  MAPE: 12.12%
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008041 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3472
[LightGBM] [Info] Number of data points in the train set: 128729, number of used features: 35
[LightGBM] [Info] Start training from score 13.779528
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

In [16]:
comparison_df_lgb = pd.DataFrame(comparison_results_lgb).T
comparison_df_lgb["rmse"] = comparison_df_lgb["rmse"].map(lambda x: f"${x:,.0f}")
comparison_df_lgb["mae"] = comparison_df_lgb["mae"].map(lambda x: f"${x:,.0f}")
comparison_df_lgb["r2"] = comparison_df_lgb["r2"].map(lambda x: f"{x:.4f}")
comparison_df_lgb["mape"] = comparison_df_lgb["mape"].map(lambda x: f"{x:.2f}%")
comparison_df_lgb


,rmse,mae,r2,mape
Baseline (with outliers),"$524,029","$182,287",0.8524,12.12%
Baseline (no outliers),"$524,029","$182,287",0.8524,12.12%
Tuned (no outliers),"$509,929","$173,741",0.8602,11.53%


In [17]:
final_comparison = {
    "XGBoost (tuned)": evaluate(
        y_val_no_outliers, xgb_tuned.predict(X_val_no_outliers), "XGB tuned"
    ),
    "LightGBM (tuned)": evaluate(
        y_val_no_outliers, lgb_tuned.predict(X_val_no_outliers), "LGBM tuned"
    ),
}

final_comparison_df = pd.DataFrame(final_comparison).T
final_comparison_df["rmse"] = final_comparison_df["rmse"].map(lambda x: f"${x:,.0f}")
final_comparison_df["mae"] = final_comparison_df["mae"].map(lambda x: f"${x:,.0f}")
final_comparison_df["r2"] = final_comparison_df["r2"].map(lambda x: f"{x:.4f}")
final_comparison_df["mape"] = final_comparison_df["mape"].map(lambda x: f"{x:.2f}%")
final_comparison_df


 XGB tuned | RMSE: $505,134  MAE: $172,691  R2: 0.8629  MAPE: 11.49%
LGBM tuned | RMSE: $509,929  MAE: $173,741  R2: 0.8602  MAPE: 11.53%


,rmse,mae,r2,mape
XGBoost (tuned),"$505,134","$172,691",0.8629,11.49%
LightGBM (tuned),"$509,929","$173,741",0.8602,11.53%


In [18]:
RANDOM_STATE = 20260805


def evaluate(y_true_log, y_pred_log, label=""):
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)

    rmse = root_mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(
        f"{label:>10} | RMSE: ${rmse:,.0f}  MAE: ${mae:,.0f}  R2: {r2:.4f}  MAPE: {mape:.2f}%"
    )
    return {"rmse": rmse, "mae": mae, "r2": r2, "mape": mape}


from sklearn.tree import DecisionTreeRegressor

dt_check = DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE)
dt_check.fit(X_train_no_outliers, y_train_no_outliers)
evaluate(y_val_no_outliers, dt_check.predict(X_val_no_outliers), "DT check")


  DT check | RMSE: $638,380  MAE: $234,207  R2: 0.7810  MAPE: 15.67%


{'rmse': 638379.7637525207,
 'mae': 234207.16948234293,
 'r2': 0.7809717460398685,
 'mape': np.float64(15.671960274525357)}

In [19]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import make_scorer


def r2_on_price_scale(y_true_log, y_pred_log):
    return r2_score(np.exp(y_true_log), np.exp(y_pred_log))


price_scale_r2 = make_scorer(r2_on_price_scale, greater_is_better=True)
tscv = TimeSeriesSplit(n_splits=5)

param_grid_dt = {"max_depth": [3, 5, 7, 10, 15, 20, None]}

grid_search_dt = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=RANDOM_STATE),
    param_grid=param_grid_dt,
    cv=tscv,
    scoring=price_scale_r2,
    n_jobs=-1,
)
grid_search_dt.fit(X_train_no_outliers, y_train_no_outliers)
print(f"Best max_depth: {grid_search_dt.best_params_}")

dt_tuned = DecisionTreeRegressor(
    max_depth=grid_search_dt.best_params_["max_depth"],
    random_state=RANDOM_STATE,
)
dt_tuned.fit(X_train_no_outliers, y_train_no_outliers)
evaluate(
    y_val_no_outliers,
    dt_tuned.predict(X_val_no_outliers),
    "DT tuned",
)


Best max_depth: {'max_depth': 7}
  DT tuned | RMSE: $662,754  MAE: $258,839  R2: 0.7639  MAPE: 17.56%


{'rmse': 662754.1979533253,
 'mae': 258838.67571208847,
 'r2': 0.7639266884151987,
 'mape': np.float64(17.56174739131495)}

In [20]:
from sklearn.ensemble import RandomForestRegressor

param_grid_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 15, 20, None],
    "max_features": ["sqrt", "log2", 0.5],
}

grid_search_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=param_grid_rf,
    cv=tscv,
    scoring=price_scale_r2,
    n_jobs=1,
)
grid_search_rf.fit(X_train_no_outliers, y_train_no_outliers)
print(f"Best params: {grid_search_rf.best_params_}")

rf_tuned = RandomForestRegressor(
    **grid_search_rf.best_params_, random_state=RANDOM_STATE, n_jobs=-1
)
rf_tuned.fit(X_train_no_outliers, y_train_no_outliers)
evaluate(
    y_val_no_outliers,
    rf_tuned.predict(X_val_no_outliers),
    "RF tuned",
)


Best params: {'max_depth': None, 'max_features': 0.5, 'n_estimators': 300}
  RF tuned | RMSE: $541,342  MAE: $180,958  R2: 0.8425  MAPE: 11.88%


{'rmse': 541341.6985923857,
 'mae': 180957.96058428512,
 'r2': 0.8424984325140061,
 'mape': np.float64(11.880774798304957)}

In [21]:
final_train_val = {
    "Decision Tree (tuned)": {
        "train_r2": r2_score(
            np.exp(y_train_no_outliers), np.exp(dt_tuned.predict(X_train_no_outliers))
        ),
        "val_r2": r2_score(
            np.exp(y_val_no_outliers), np.exp(dt_tuned.predict(X_val_no_outliers))
        ),
    },
    "Random Forest (tuned)": {
        "train_r2": r2_score(
            np.exp(y_train_no_outliers), np.exp(rf_tuned.predict(X_train_no_outliers))
        ),
        "val_r2": r2_score(
            np.exp(y_val_no_outliers), np.exp(rf_tuned.predict(X_val_no_outliers))
        ),
    },
    "XGBoost (tuned)": {
        "train_r2": r2_score(
            np.exp(y_train_no_outliers), np.exp(xgb_tuned.predict(X_train_no_outliers))
        ),
        "val_r2": r2_score(
            np.exp(y_val_no_outliers), np.exp(xgb_tuned.predict(X_val_no_outliers))
        ),
    },
    "LightGBM (tuned)": {
        "train_r2": r2_score(
            np.exp(y_train_no_outliers), np.exp(lgb_tuned.predict(X_train_no_outliers))
        ),
        "val_r2": r2_score(
            np.exp(y_val_no_outliers), np.exp(lgb_tuned.predict(X_val_no_outliers))
        ),
    },
}

train_val_df = pd.DataFrame(final_train_val).T
train_val_df["gap"] = train_val_df["train_r2"] - train_val_df["val_r2"]
train_val_df = train_val_df.round(4)
train_val_df


,train_r2,val_r2,gap
Decision Tree (tuned),0.7948,0.7639,0.0308
Random Forest (tuned),0.9723,0.8425,0.1298
XGBoost (tuned),0.9479,0.8629,0.0850
LightGBM (tuned),0.9398,0.8602,0.0796


## Model Improvement Experiments

### 1. Weighted Training (Luxury Segment)


In [23]:
# Search over luxury-segment weight multipliers to find the best one on Val
weight_options = [1, 2, 3, 4, 5, 8, 10]
weight_search_results = []
train_price = np.exp(y_train_no_outliers)
sample_weight = np.where(train_price > 2_000_000, 4, 1)

for w in weight_options:
    sample_weight_w = np.where(train_price > 2_000_000, w, 1)

    xgb_w_search = xgb.XGBRegressor(
        max_depth=6,
        learning_rate=0.1,
        n_estimators=600,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    xgb_w_search.fit(
        X_train_no_outliers, y_train_no_outliers, sample_weight=sample_weight_w
    )

    val_pred_log = xgb_w_search.predict(X_val_no_outliers)
    metrics = evaluate(y_val_no_outliers, val_pred_log, f"weight={w}")

    y_val_price = np.exp(y_val_no_outliers)
    val_pred_price = np.exp(val_pred_log)
    mdape = np.median(np.abs((y_val_price - val_pred_price) / y_val_price)) * 100

    weight_search_results.append({"weight": w, **metrics, "mdape": mdape})

weight_search_df = pd.DataFrame(weight_search_results)
print("\n=== Weight Multiplier Search (Validation Set) ===")
print(weight_search_df.to_string(index=False))

best_weight_row = weight_search_df.loc[weight_search_df["r2"].idxmax()]
print(f"\nBest weight by R²: {best_weight_row['weight']:.0f}")


  weight=1 | RMSE: $505,134  MAE: $172,691  R2: 0.8629  MAPE: 11.49%
  weight=2 | RMSE: $511,743  MAE: $173,279  R2: 0.8593  MAPE: 11.63%
  weight=3 | RMSE: $501,185  MAE: $173,609  R2: 0.8650  MAPE: 11.78%
  weight=4 | RMSE: $495,478  MAE: $174,707  R2: 0.8681  MAPE: 11.95%
  weight=5 | RMSE: $500,763  MAE: $176,453  R2: 0.8652  MAPE: 12.05%
  weight=8 | RMSE: $498,662  MAE: $178,912  R2: 0.8664  MAPE: 12.35%
 weight=10 | RMSE: $505,748  MAE: $181,537  R2: 0.8625  MAPE: 12.51%

=== Weight Multiplier Search (Validation Set) ===
 weight          rmse           mae       r2      mape    mdape
      1 505133.768758 172690.943985 0.862863 11.490449 7.879392
      2 511742.742635 173278.997221 0.859251 11.630168 7.945532
      3 501185.474552 173608.840747 0.864998 11.782980 8.005093
      4 495477.670307 174706.799861 0.868056 11.948558 8.125489
      5 500763.044363 176453.047536 0.865226 12.049691 8.169496
      8 498662.089445 178911.573954 0.866354 12.347104 8.274417
     10 505747.824

In [24]:
# Weighted training experiment: give higher weight to luxury properties (>$2M)
# to address the sparse-sample problem observed in the price band analysis


xgb_weighted = xgb.XGBRegressor(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=600,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_weighted.fit(X_train_no_outliers, y_train_no_outliers, sample_weight=sample_weight)

print("=== Weighted vs. Unweighted (Validation Set) ===")
evaluate(y_val_no_outliers, xgb_tuned.predict(X_val_no_outliers), "XGB (no weight)")
evaluate(y_val_no_outliers, xgb_weighted.predict(X_val_no_outliers), "XGB (weighted)")


=== Weighted vs. Unweighted (Validation Set) ===
XGB (no weight) | RMSE: $505,134  MAE: $172,691  R2: 0.8629  MAPE: 11.49%
XGB (weighted) | RMSE: $495,478  MAE: $174,707  R2: 0.8681  MAPE: 11.95%


{'rmse': 495477.6703074288,
 'mae': 174706.799861076,
 'r2': 0.8680558646340908,
 'mape': np.float64(11.948557735985336)}

In [25]:
def price_band_analysis(model, X_test, y_test_log, model_name=""):
    y_true = np.exp(y_test_log)
    y_pred = np.exp(model.predict(X_test))

    bands = pd.cut(
        y_true,
        bins=[0, 500_000, 1_000_000, 2_000_000, np.inf],
        labels=["<$500K", "$500K-$1M", "$1M-$2M", "$2M+"],
    )

    results = []
    for band in bands.cat.categories:
        mask = bands == band
        if mask.sum() == 0:
            continue
        yt, yp = y_true[mask], y_pred[mask]
        rmse = root_mean_squared_error(yt, yp)
        mae = mean_absolute_error(yt, yp)
        mape = np.mean(np.abs((yt - yp) / yt)) * 100
        mdape = np.median(np.abs((yt - yp) / yt)) * 100
        results.append(
            {
                "price_band": band,
                "n_listings": mask.sum(),
                "rmse": rmse,
                "mae": mae,
                "mape": mape,
                "mdape": mdape,
            }
        )

    band_df = pd.DataFrame(results)
    print(f"=== {model_name} — Performance by Price Band ===")
    print(band_df.to_string(index=False))
    return band_df


xgb_base_band_val = price_band_analysis(
    xgb_tuned, X_val_no_outliers, y_val_no_outliers, "XGB No Weight (Val)"
)

xgb_weighted_band_val = price_band_analysis(
    xgb_weighted, X_val_no_outliers, y_val_no_outliers, "XGB Weighted (Val)"
)


=== XGB No Weight (Val) — Performance by Price Band ===
price_band  n_listings         rmse           mae      mape     mdape
    <$500K        6204 7.970871e+04  49672.278321 14.712957  7.813874
 $500K-$1M       18254 1.003795e+05  67101.654188  8.996045  6.291356
   $1M-$2M       13006 2.294773e+05 162253.122849 11.479575  8.954743
      $2M+        6031 1.299844e+06 641334.452432 15.748757 12.615065
=== XGB Weighted (Val) — Performance by Price Band ===
price_band  n_listings         rmse           mae      mape     mdape
    <$500K        6204 8.338758e+04  51176.647104 15.149239  8.041898
 $500K-$1M       18254 1.061802e+05  69208.993519  9.273874  6.417750
   $1M-$2M       13006 2.588252e+05 183433.372440 12.979765  9.965846
      $2M+        6031 1.258875e+06 602271.087992 14.527692 11.233773


In [26]:
# Search over luxury-segment weight multipliers for LightGBM (separately from
# XGBoost, since the two models may respond differently to sample weighting)
weight_options = [1, 2, 3, 4, 5, 8, 10]
lgb_weight_search_results = []

for w in weight_options:
    sample_weight_w = np.where(train_price > 2_000_000, w, 1)

    lgb_w_search = lgb.LGBMRegressor(
        max_depth=7,
        learning_rate=0.15,
        n_estimators=600,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    lgb_w_search.fit(
        X_train_no_outliers, y_train_no_outliers, sample_weight=sample_weight_w
    )

    val_pred_log = lgb_w_search.predict(X_val_no_outliers)
    metrics = evaluate(y_val_no_outliers, val_pred_log, f"weight={w}")

    y_val_price = np.exp(y_val_no_outliers)
    val_pred_price = np.exp(val_pred_log)
    mdape = np.median(np.abs((y_val_price - val_pred_price) / y_val_price)) * 100

    lgb_weight_search_results.append({"weight": w, **metrics, "mdape": mdape})

lgb_weight_search_df = pd.DataFrame(lgb_weight_search_results)
print("\n=== LightGBM Weight Multiplier Search (Validation Set) ===")
print(lgb_weight_search_df.to_string(index=False))

best_lgb_weight_row = lgb_weight_search_df.loc[lgb_weight_search_df["r2"].idxmax()]
print(f"\nBest weight by R²: {best_lgb_weight_row['weight']:.0f}")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003924 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3472
[LightGBM] [Info] Number of data points in the train set: 128729, number of used features: 35
[LightGBM] [Info] Start training from score 13.779528
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

In [27]:
train_price = np.exp(y_train_no_outliers)
sample_weight_lgb = np.where(train_price > 2_000_000, 2, 1)

lgb_weighted = lgb.LGBMRegressor(
    max_depth=7,
    learning_rate=0.15,
    n_estimators=600,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
lgb_weighted.fit(
    X_train_no_outliers, y_train_no_outliers, sample_weight=sample_weight_lgb
)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004474 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3472
[LightGBM] [Info] Number of data points in the train set: 128729, number of used features: 35
[LightGBM] [Info] Start training from score 13.914945
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,7
,learning_rate,0.15
,n_estimators,600
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [28]:
lgb_base_band_val = price_band_analysis(
    lgb_tuned, X_val_no_outliers, y_val_no_outliers, "LGBM No Weight (Val)"
)
lgb_weighted_band_val = price_band_analysis(
    lgb_weighted, X_val_no_outliers, y_val_no_outliers, "LGBM Weighted (Val)"
)


=== LGBM No Weight (Val) — Performance by Price Band ===
price_band  n_listings         rmse           mae      mape     mdape
    <$500K        6204 8.010447e+04  50203.366550 14.833669  7.910311
 $500K-$1M       18254 1.002976e+05  66926.753501  8.979274  6.313801
   $1M-$2M       13006 2.286607e+05 162704.882795 11.506074  9.007119
      $2M+        6031 1.313579e+06 647914.245587 15.881369 12.719111
=== LGBM Weighted (Val) — Performance by Price Band ===
price_band  n_listings         rmse           mae      mape     mdape
    <$500K        6204 8.166940e+04  50700.152999 14.927587  7.894663
 $500K-$1M       18254 1.030701e+05  68266.999273  9.150290  6.450986
   $1M-$2M       13006 2.433634e+05 171901.938415 12.154111  9.355982
      $2M+        6031 1.293818e+06 629457.112567 15.277661 12.039074


### Ensemble (XGB + LGBM) 

In [29]:
xgb_w_pred_price = np.exp(xgb_weighted.predict(X_val_no_outliers))
lgb_w_pred_price = np.exp(lgb_weighted.predict(X_val_no_outliers))
y_true_price = np.exp(y_val_no_outliers)

ensemble_pred_price = 0.5 * xgb_w_pred_price + 0.5 * lgb_w_pred_price

rmse = root_mean_squared_error(y_true_price, ensemble_pred_price)
mae = mean_absolute_error(y_true_price, ensemble_pred_price)
r2 = r2_score(y_true_price, ensemble_pred_price)
mape = np.mean(np.abs((y_true_price - ensemble_pred_price) / y_true_price)) * 100
mdape = np.median(np.abs((y_true_price - ensemble_pred_price) / y_true_price)) * 100

print(
    f"Ensemble (weighted, 50/50, Val) | RMSE: ${rmse:,.0f}  MAE: ${mae:,.0f}  R2: {r2:.4f}  MAPE: {mape:.2f}%  MdAPE: {mdape:.2f}%"
)


Ensemble (weighted, 50/50, Val) | RMSE: $492,785  MAE: $171,903  R2: 0.8695  MAPE: 11.68%  MdAPE: 7.93%


In [30]:
best_r2 = -np.inf
best_w = None
for w in np.arange(0, 1.05, 0.1):
    pred = w * xgb_w_pred_price + (1 - w) * lgb_w_pred_price
    r2_w = r2_score(y_true_price, pred)
    print(f"XGB weight={w:.1f} | R²={r2_w:.4f}")
    if r2_w > best_r2:
        best_r2 = r2_w
        best_w = w

print(f"\nBest XGB weight: {best_w:.1f}, R²: {best_r2:.4f}")


XGB weight=0.0 | R²=0.8628
XGB weight=0.1 | R²=0.8648
XGB weight=0.2 | R²=0.8665
XGB weight=0.3 | R²=0.8678
XGB weight=0.4 | R²=0.8688
XGB weight=0.5 | R²=0.8695
XGB weight=0.6 | R²=0.8698
XGB weight=0.7 | R²=0.8699
XGB weight=0.8 | R²=0.8696
XGB weight=0.9 | R²=0.8690
XGB weight=1.0 | R²=0.8681

Best XGB weight: 0.7, R²: 0.8699


In [31]:
def price_band_analysis_ensemble(pred_price, y_true_log, model_name=""):
    y_true = np.exp(y_true_log)

    bands = pd.cut(
        y_true,
        bins=[0, 500_000, 1_000_000, 2_000_000, np.inf],
        labels=["<$500K", "$500K-$1M", "$1M-$2M", "$2M+"],
    )

    results = []
    for band in bands.cat.categories:
        mask = bands == band
        if mask.sum() == 0:
            continue
        yt, yp = y_true[mask], pred_price[mask]
        rmse = root_mean_squared_error(yt, yp)
        mae = mean_absolute_error(yt, yp)
        mape = np.mean(np.abs((yt - yp) / yt)) * 100
        mdape = np.median(np.abs((yt - yp) / yt)) * 100
        results.append(
            {
                "price_band": band,
                "n_listings": mask.sum(),
                "rmse": rmse,
                "mae": mae,
                "mape": mape,
                "mdape": mdape,
            }
        )

    band_df = pd.DataFrame(results)
    print(f"=== {model_name} — Performance by Price Band ===")
    print(band_df.to_string(index=False))
    return band_df


final_ensemble_pred = best_w * xgb_w_pred_price + (1 - best_w) * lgb_w_pred_price
ensemble_band_results = price_band_analysis_ensemble(
    final_ensemble_pred, y_val_no_outliers, "Ensemble (best weight, Val)"
)

=== Ensemble (best weight, Val) — Performance by Price Band ===
price_band  n_listings         rmse           mae      mape     mdape
    <$500K        6204 8.216407e+04  50621.645833 14.971087  7.818752
 $500K-$1M       18254 1.041541e+05  68169.420330  9.135145  6.364882
   $1M-$2M       13006 2.511110e+05 178013.471529 12.594944  9.565328
      $2M+        6031 1.253077e+06 600716.948331 14.521035 11.265037


### Regularization

In [32]:
# v1: baseline weighted model (for comparison)
train_r2 = r2_score(
    np.exp(y_train_no_outliers), np.exp(xgb_weighted.predict(X_train_no_outliers))
)
val_r2 = r2_score(
    np.exp(y_val_no_outliers), np.exp(xgb_weighted.predict(X_val_no_outliers))
)
print(
    f"Baseline (weighted, no regularization) | Train R²: {train_r2:.4f}, Val R²: {val_r2:.4f}, Gap: {train_r2 - val_r2:.4f}"
)


Baseline (weighted, no regularization) | Train R²: 0.9635, Val R²: 0.8681, Gap: 0.0954


In [33]:
lgb_reg_v3 = lgb.LGBMRegressor(
    max_depth=7,
    learning_rate=0.15,
    n_estimators=600,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    reg_lambda=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
lgb_reg_v3.fit(X_train_no_outliers, y_train_no_outliers, sample_weight=sample_weight)

train_r2 = r2_score(
    np.exp(y_train_no_outliers), np.exp(lgb_reg_v3.predict(X_train_no_outliers))
)
val_r2 = r2_score(
    np.exp(y_val_no_outliers), np.exp(lgb_reg_v3.predict(X_val_no_outliers))
)
print(
    f"LGBM v3 (regularized) | Train R²: {train_r2:.4f}, Val R²: {val_r2:.4f}, Gap: {train_r2 - val_r2:.4f}"
)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003670 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3472
[LightGBM] [Info] Number of data points in the train set: 128729, number of used features: 35
[LightGBM] [Info] Start training from score 14.110159
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

In [34]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "max_depth": [3, 4, 5, 6, 7],
    "learning_rate": [0.03, 0.05, 0.08, 0.1, 0.15],
    "n_estimators": [200, 300, 400, 600],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 10],
    "reg_alpha": [0, 0.1, 0.5, 1, 2],
    "reg_lambda": [1, 1.5, 2, 3],
}

random_search_xgb = RandomizedSearchCV(
    estimator=xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=50,
    cv=tscv,
    scoring=price_scale_r2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

random_search_xgb.fit(
    X_train_no_outliers,
    y_train_no_outliers,
    sample_weight=sample_weight,
)

print(f"Best params: {random_search_xgb.best_params_}")
print(f"Best CV R²: {random_search_xgb.best_score_:.4f}")


Fitting 5 folds for each of 50 candidates, totalling 250 fits


/opt/base-uv/.venv/lib/python3.13/site-packages/sklearn/model_selection/_search.py:883: UserWarning: The scoring make_scorer(r2_on_price_scale, response_method='predict') does not support sample_weight, which may lead to statistically incorrect results when fitting RandomizedSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamm...
         

Best params: {'subsample': 0.6, 'reg_lambda': 2, 'reg_alpha': 2, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 6, 'learning_rate': 0.15, 'colsample_bytree': 0.8}
Best CV R²: 0.8634


In [35]:
xgb_auto = xgb.XGBRegressor(
    subsample=0.8,
    reg_lambda=3,
    reg_alpha=2,
    n_estimators=600,
    min_child_weight=10,
    max_depth=4,
    learning_rate=0.08,
    colsample_bytree=0.7,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_auto.fit(X_train_no_outliers, y_train_no_outliers, sample_weight=sample_weight)

train_r2 = r2_score(
    np.exp(y_train_no_outliers), np.exp(xgb_auto.predict(X_train_no_outliers))
)
val_r2 = r2_score(
    np.exp(y_val_no_outliers), np.exp(xgb_auto.predict(X_val_no_outliers))
)
print(
    f"Auto-tuned XGB | Train R²: {train_r2:.4f}, Val R²: {val_r2:.4f}, Gap: {train_r2 - val_r2:.4f}"
)


Auto-tuned XGB | Train R²: 0.9043, Val R²: 0.8571, Gap: 0.0472


In [36]:
param_dist_lgb = {
    "max_depth": [3, 4, 5, 6, 7],
    "learning_rate": [0.03, 0.05, 0.08, 0.1, 0.15],
    "n_estimators": [200, 300, 400, 600],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_samples": [5, 10, 20, 30],
    "reg_alpha": [0, 0.1, 0.5, 1, 2],
    "reg_lambda": [1, 1.5, 2, 3],
}

random_search_lgb = RandomizedSearchCV(
    estimator=lgb.LGBMRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_dist_lgb,
    n_iter=50,
    cv=tscv,
    scoring=price_scale_r2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

random_search_lgb.fit(
    X_train_no_outliers,
    y_train_no_outliers,
    sample_weight=sample_weight,
)

print(f"Best params: {random_search_lgb.best_params_}")
print(f"Best CV R²: {random_search_lgb.best_score_:.4f}")


Fitting 5 folds for each of 50 candidates, totalling 250 fits


/opt/base-uv/.venv/lib/python3.13/site-packages/sklearn/model_selection/_search.py:883: UserWarning: The scoring make_scorer(r2_on_price_scale, response_method='predict') does not support sample_weight, which may lead to statistically incorrect results when fitting RandomizedSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
                   estimator=LGBMRegressor(n_jobs=-1, random_state=20260805),
                   n_iter=50, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.6, 0.7, 0.8, 0.9,
                                                             1.0],
                                        'learning_rate': [0.03, 0.05, 0.08, 0.1,
                                                          0.15],
                                        'max_depth': [3, 4, 5, 6, 7],
                                        'min_child_samples': [5, 10, 20, 30],
                                        'n_estimators': [200, 300, 400, 600],


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004646 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3472
[LightGBM] [Info] Number of data points in the train set: 128729, number of used features: 35
[LightGBM] [Info] Start training from score 14.110159
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

In [37]:
lgb_auto = lgb.LGBMRegressor(
    subsample=0.7,
    reg_lambda=1,
    reg_alpha=2,
    n_estimators=300,
    min_child_samples=10,
    max_depth=7,
    learning_rate=0.1,
    colsample_bytree=0.7,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
lgb_auto.fit(X_train_no_outliers, y_train_no_outliers, sample_weight=sample_weight)

train_r2 = r2_score(
    np.exp(y_train_no_outliers), np.exp(lgb_auto.predict(X_train_no_outliers))
)
val_r2 = r2_score(
    np.exp(y_val_no_outliers), np.exp(lgb_auto.predict(X_val_no_outliers))
)
print(
    f"Auto-tuned LGBM | Train R²: {train_r2:.4f}, Val R²: {val_r2:.4f}, Gap: {train_r2 - val_r2:.4f}"
)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003354 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3472
[LightGBM] [Info] Number of data points in the train set: 128729, number of used features: 35
[LightGBM] [Info] Start training from score 14.110159
Auto-tuned LGBM | Train R²: 0.9178, Val R²: 0.8573, Gap: 0.0605


In [38]:
xgb_auto_val_pred = np.exp(xgb_auto.predict(X_val_no_outliers))
lgb_auto_val_pred = np.exp(lgb_auto.predict(X_val_no_outliers))
y_val_price = np.exp(y_val_no_outliers)

best_r2 = -np.inf
best_w = None
for w in np.arange(0, 1.05, 0.1):
    pred = w * xgb_auto_val_pred + (1 - w) * lgb_auto_val_pred
    r2_w = r2_score(y_val_price, pred)
    print(f"XGB weight={w:.1f} | R²={r2_w:.4f}")
    if r2_w > best_r2:
        best_r2 = r2_w
        best_w = w

print(f"\nBest XGB weight: {best_w:.1f}, R²: {best_r2:.4f}")


XGB weight=0.0 | R²=0.8573
XGB weight=0.1 | R²=0.8582
XGB weight=0.2 | R²=0.8589
XGB weight=0.3 | R²=0.8593
XGB weight=0.4 | R²=0.8596
XGB weight=0.5 | R²=0.8597
XGB weight=0.6 | R²=0.8596
XGB weight=0.7 | R²=0.8593
XGB weight=0.8 | R²=0.8587
XGB weight=0.9 | R²=0.8580
XGB weight=1.0 | R²=0.8571

Best XGB weight: 0.5, R²: 0.8597


In [39]:
xgb_auto_train_pred = np.exp(xgb_auto.predict(X_train_no_outliers))
lgb_auto_train_pred = np.exp(lgb_auto.predict(X_train_no_outliers))
y_train_price = np.exp(y_train_no_outliers)

ensemble_train_pred = best_w * xgb_auto_train_pred + (1 - best_w) * lgb_auto_train_pred
ensemble_val_pred = best_w * xgb_auto_val_pred + (1 - best_w) * lgb_auto_val_pred

train_r2_new = r2_score(y_train_price, ensemble_train_pred)
val_r2_new = r2_score(y_val_price, ensemble_val_pred)

print(
    f"New Ensemble (auto-tuned) | Train R²: {train_r2_new:.4f}, Val R²: {val_r2_new:.4f}, Gap: {train_r2_new - val_r2_new:.4f}"
)


New Ensemble (auto-tuned) | Train R²: 0.9130, Val R²: 0.8597, Gap: 0.0533


## New Ensemble (Auto-Tuned)

In [40]:
xgb_auto_test_pred = np.exp(xgb_auto.predict(X_test_no_outliers))
lgb_auto_test_pred = np.exp(lgb_auto.predict(X_test_no_outliers))
y_test_price = np.exp(y_test_no_outliers)

final_ensemble_test_pred = (
    best_w * xgb_auto_test_pred + (1 - best_w) * lgb_auto_test_pred
)

rmse = root_mean_squared_error(y_test_price, final_ensemble_test_pred)
mae = mean_absolute_error(y_test_price, final_ensemble_test_pred)
r2 = r2_score(y_test_price, final_ensemble_test_pred)
mape = np.mean(np.abs((y_test_price - final_ensemble_test_pred) / y_test_price)) * 100
mdape = (
    np.median(np.abs((y_test_price - final_ensemble_test_pred) / y_test_price)) * 100
)

print(
    f"Final Ensemble (Test Set) | RMSE: ${rmse:,.0f}  MAE: ${mae:,.0f}  R2: {r2:.4f}  MAPE: {mape:.2f}%  MdAPE: {mdape:.2f}%"
)


Final Ensemble (Test Set) | RMSE: $567,413  MAE: $190,336  R2: 0.8415  MAPE: 12.79%  MdAPE: 8.72%


In [41]:
def price_band_analysis_ensemble(pred_price, y_true_log, model_name=""):
    y_true = np.exp(y_true_log)

    bands = pd.cut(
        y_true,
        bins=[0, 500_000, 1_000_000, 2_000_000, np.inf],
        labels=["<$500K", "$500K-$1M", "$1M-$2M", "$2M+"],
    )

    results = []
    for band in bands.cat.categories:
        mask = bands == band
        if mask.sum() == 0:
            continue
        yt, yp = y_true[mask], pred_price[mask]
        rmse = root_mean_squared_error(yt, yp)
        mae = mean_absolute_error(yt, yp)
        mape = np.mean(np.abs((yt - yp) / yt)) * 100
        mdape = np.median(np.abs((yt - yp) / yt)) * 100
        results.append(
            {
                "price_band": band,
                "n_listings": mask.sum(),
                "rmse": rmse,
                "mae": mae,
                "mape": mape,
                "mdape": mdape,
            }
        )

    band_df = pd.DataFrame(results)
    print(f"=== {model_name} — Performance by Price Band ===")
    print(band_df.to_string(index=False))
    return band_df


final_band_results = price_band_analysis_ensemble(
    final_ensemble_test_pred, y_test_no_outliers, "Final Ensemble (Auto-Tuned)"
)


=== Final Ensemble (Auto-Tuned) — Performance by Price Band ===
price_band  n_listings         rmse           mae      mape     mdape
    <$500K        1782 8.314735e+04  54879.255281 16.712013  9.302007
 $500K-$1M        5316 1.135478e+05  74460.789767  9.881946  6.830678
   $1M-$2M        3908 2.722747e+05 197358.328123 13.930517 10.580085
      $2M+        1784 1.449305e+06 655546.914297 15.032849 11.581807


In [45]:
final_summary_rows = []


def dollar_metrics(y_true_log, y_pred_log):
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    mdape = np.median(np.abs((y_true - y_pred) / y_true)) * 100
    return mape, mdape


def get_r2(model, X, y_log):
    return r2_score(np.exp(y_log), np.exp(model.predict(X)))


# 1. XGB / LGBM baseline (tuned, no weighting, no reg)
xgb_base_mape, xgb_base_mdape = dollar_metrics(
    y_test_no_outliers, xgb_tuned.predict(X_test_no_outliers)
)
lgb_base_mape, lgb_base_mdape = dollar_metrics(
    y_test_no_outliers, lgb_tuned.predict(X_test_no_outliers)
)

final_summary_rows.append(
    {
        "stage": "XGB baseline (tuned)",
        "train_r2": get_r2(xgb_tuned, X_train_no_outliers, y_train_no_outliers),
        "val_r2": get_r2(xgb_tuned, X_val_no_outliers, y_val_no_outliers),
        "test_r2": get_r2(xgb_tuned, X_test_no_outliers, y_test_no_outliers),
        "mape": xgb_base_mape,
        "mdape": xgb_base_mdape,
    }
)
final_summary_rows.append(
    {
        "stage": "LGBM baseline (tuned)",
        "train_r2": get_r2(lgb_tuned, X_train_no_outliers, y_train_no_outliers),
        "val_r2": get_r2(lgb_tuned, X_val_no_outliers, y_val_no_outliers),
        "test_r2": get_r2(lgb_tuned, X_test_no_outliers, y_test_no_outliers),
        "mape": lgb_base_mape,
        "mdape": lgb_base_mdape,
    }
)

# 2. Weighted (no reg)
xgb_w_mape, xgb_w_mdape = dollar_metrics(
    y_test_no_outliers, xgb_weighted.predict(X_test_no_outliers)
)
lgb_w_mape, lgb_w_mdape = dollar_metrics(
    y_test_no_outliers, lgb_weighted.predict(X_test_no_outliers)
)

final_summary_rows.append(
    {
        "stage": "XGB weighted (no reg)",
        "train_r2": get_r2(xgb_weighted, X_train_no_outliers, y_train_no_outliers),
        "val_r2": get_r2(xgb_weighted, X_val_no_outliers, y_val_no_outliers),
        "test_r2": get_r2(xgb_weighted, X_test_no_outliers, y_test_no_outliers),
        "mape": xgb_w_mape,
        "mdape": xgb_w_mdape,
    }
)
final_summary_rows.append(
    {
        "stage": "LGBM weighted (no reg)",
        "train_r2": get_r2(lgb_weighted, X_train_no_outliers, y_train_no_outliers),
        "val_r2": get_r2(lgb_weighted, X_val_no_outliers, y_val_no_outliers),
        "test_r2": get_r2(lgb_weighted, X_test_no_outliers, y_test_no_outliers),
        "mape": lgb_w_mape,
        "mdape": lgb_w_mdape,
    }
)

# 3. Ensemble (weighted, no reg) — best_w=0.3 from val search
xgb_w_train_pred = np.exp(xgb_weighted.predict(X_train_no_outliers))
lgb_w_train_pred = np.exp(lgb_weighted.predict(X_train_no_outliers))
xgb_w_val_pred_ = np.exp(xgb_weighted.predict(X_val_no_outliers))
lgb_w_val_pred_ = np.exp(lgb_weighted.predict(X_val_no_outliers))
xgb_w_test_pred = np.exp(xgb_weighted.predict(X_test_no_outliers))
lgb_w_test_pred = np.exp(lgb_weighted.predict(X_test_no_outliers))

y_train_price_ = np.exp(y_train_no_outliers)
y_val_price_ = np.exp(y_val_no_outliers)
y_test_price_ = np.exp(y_test_no_outliers)

ens_train_pred = best_w * xgb_w_train_pred + (1 - best_w) * lgb_w_train_pred
ens_val_pred = best_w * xgb_w_val_pred_ + (1 - best_w) * lgb_w_val_pred_
ens_test_pred = best_w * xgb_w_test_pred + (1 - best_w) * lgb_w_test_pred

ens_mape = np.mean(np.abs((y_test_price_ - ens_test_pred) / y_test_price_)) * 100
ens_mdape = np.median(np.abs((y_test_price_ - ens_test_pred) / y_test_price_)) * 100

final_summary_rows.append(
    {
        "stage": "Ensemble (weighted, no reg)",
        "train_r2": r2_score(y_train_price_, ens_train_pred),
        "val_r2": r2_score(y_val_price_, ens_val_pred),
        "test_r2": r2_score(y_test_price_, ens_test_pred),
        "mape": ens_mape,
        "mdape": ens_mdape,
    }
)

# 4. Auto-tuned (weighted + regularized)
xgb_auto_mape, xgb_auto_mdape = dollar_metrics(
    y_test_no_outliers, xgb_auto.predict(X_test_no_outliers)
)
lgb_auto_mape, lgb_auto_mdape = dollar_metrics(
    y_test_no_outliers, lgb_auto.predict(X_test_no_outliers)
)

final_summary_rows.append(
    {
        "stage": "XGB auto-tuned (weighted + regularized)",
        "train_r2": get_r2(xgb_auto, X_train_no_outliers, y_train_no_outliers),
        "val_r2": get_r2(xgb_auto, X_val_no_outliers, y_val_no_outliers),
        "test_r2": get_r2(xgb_auto, X_test_no_outliers, y_test_no_outliers),
        "mape": xgb_auto_mape,
        "mdape": xgb_auto_mdape,
    }
)
final_summary_rows.append(
    {
        "stage": "LGBM auto-tuned (weighted + regularized)",
        "train_r2": get_r2(lgb_auto, X_train_no_outliers, y_train_no_outliers),
        "val_r2": get_r2(lgb_auto, X_val_no_outliers, y_val_no_outliers),
        "test_r2": get_r2(lgb_auto, X_test_no_outliers, y_test_no_outliers),
        "mape": lgb_auto_mape,
        "mdape": lgb_auto_mdape,
    }
)

# 5. Final Ensemble (auto-tuned)
final_ens_mape = (
    np.mean(np.abs((y_test_price - final_ensemble_test_pred) / y_test_price)) * 100
)
final_ens_mdape = (
    np.median(np.abs((y_test_price - final_ensemble_test_pred) / y_test_price)) * 100
)

final_summary_rows.append(
    {
        "stage": "Final Ensemble (auto-tuned)",
        "train_r2": train_r2_new,
        "val_r2": val_r2_new,
        "test_r2": r2_score(y_test_price, final_ensemble_test_pred),
        "mape": final_ens_mape,
        "mdape": final_ens_mdape,
    }
)

final_summary_table = pd.DataFrame(final_summary_rows)
for col in ["train_r2", "val_r2", "test_r2", "mape", "mdape"]:
    final_summary_table[col] = final_summary_table[col].round(4)
final_summary_table["gap"] = (
    final_summary_table["train_r2"] - final_summary_table["val_r2"]
).round(4)
final_summary_table


,stage,train_r2,val_r2,test_r2,mape,mdape,gap
0,XGB baseline (tuned),0.9479,0.8629,0.8426,11.4947,7.9152,0.0850
1,LGBM baseline (tuned),0.9398,0.8602,0.8484,11.5622,7.9645,0.0796
2,XGB weighted (no reg),0.9635,0.8681,0.8373,12.0644,8.2518,0.0954
3,LGBM weighted (no reg),0.9497,0.8628,0.8482,11.7697,8.0817,0.0869
4,"Ensemble (weighted, no reg)",0.9589,0.8695,0.8463,11.7615,8.0671,0.0894
5,XGB auto-tuned (weighted + regularized),0.9043,0.8571,0.8367,13.0549,8.9219,0.0472
6,LGBM auto-tuned (weighted + regularized),0.9178,0.8573,0.8422,12.7690,8.6615,0.0605
7,Final Ensemble (auto-tuned),0.9130,0.8597,0.8415,12.7891,8.7176,0.0533


In [44]:
# LGBM Baseline price band (test set)
lgb_base_band_test = price_band_analysis(
    lgb_tuned, X_test_no_outliers, y_test_no_outliers, "LGBM Baseline (Test)"
)

# Ensemble (weighted, no reg) price band (test set)
ensemble_band_test = price_band_analysis_ensemble(
    ens_test_pred, y_test_no_outliers, "Ensemble weighted no-reg (Test)"
)


=== LGBM Baseline (Test) — Performance by Price Band ===
price_band  n_listings         rmse           mae      mape     mdape
    <$500K        1782 7.712437e+04  50514.991501 15.268815  8.450119
 $500K-$1M        5316 9.824726e+04  66650.162290  8.882928  6.203510
   $1M-$2M        3908 2.244866e+05 161992.557872 11.400010  8.887408
      $2M+        1784 1.435987e+06 689722.523574 16.198934 13.221667
=== Ensemble weighted no-reg (Test) — Performance by Price Band ===
price_band  n_listings         rmse           mae      mape     mdape
    <$500K        1782 7.846518e+04  51201.277860 15.484945  8.372904
 $500K-$1M        5316 1.010815e+05  67718.051647  9.004252  6.280850
   $1M-$2M        3908 2.413491e+05 173481.051876 12.219273  9.317159
      $2M+        1784 1.431885e+06 651096.348230 14.979658 11.736503


In [48]:
xgb_importances = pd.Series(
    xgb_weighted.feature_importances_, index=X_train_no_outliers.columns
)
lgb_importances = pd.Series(
    lgb_weighted.feature_importances_, index=X_train_no_outliers.columns
)

# Normalize each model's importances to sum to 1, so they're comparable
# despite XGBoost and LightGBM using different importance metrics by default.
xgb_norm = xgb_importances / xgb_importances.sum()
lgb_norm = lgb_importances / lgb_importances.sum()

# Combine using the same blend weight (best_w) used for the ensemble's predictions,
# so feature importance reflects the actual XGB/LGBM contribution mix.
ensemble_importances = best_w * xgb_norm + (1 - best_w) * lgb_norm

print("=== Ensemble (Weighted) Feature Importance ===")
print(ensemble_importances.sort_values(ascending=False).head(15))


=== Ensemble (Weighted) Feature Importance ===
PostalCode_encoded            0.337847
LivingArea                    0.084628
BathroomsTotalInteger         0.070584
Longitude                     0.056153
Latitude                      0.053864
City_encoded                  0.040349
YearBuilt                     0.037372
LotSizeSquareFeet             0.032326
MLSAreaMajor_encoded          0.028689
AssociationFee                0.024987
DistrictName_encoded          0.023813
LotSizeArea                   0.020361
CountyOrParish_encoded        0.019320
property_age                  0.017349
HighSchoolDistrict_encoded    0.016380
dtype: float64


In [46]:
final_summary_table.to_csv("metrics_summary.csv", index=False)
print("Saved to metrics_summary.csv")


Saved to metrics_summary.csv
